# Case study (optional): past-only lookups and the justification leak in the EBTI decisions

*Session 9, block 2, optional second example. Author: course team, licence CC-BY-4.0.*

The main practice of block 2 uses the Berlin Airbnb data ([06-case-study-airbnb-leakage.ipynb](06-case-study-airbnb-leakage.ipynb)). This notebook applies the same rules to the leaderboard task: it needs the full `train.parquet` (309,529 decisions) and takes about a minute.

Two ways in which the training data contain more than a prediction will know:

1. **`classification_justification`** is written by customs when it classifies the product. It names the heading in most rows and does not exist in the test data.
2. **Lookups by description**: many descriptions occur more than once. "The heading that this description received" is legitimate if computed from earlier decisions only, and a leak if computed from all decisions.

We measure both with a time-based validation (2017–2021 → 2022–2023). Theory: [page 2](../theory/02-aggregates-leakage-and-text-statistics.md).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
# find the repository root, wherever the notebook is started from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study" / "data").exists())
DATA = ROOT / "case-study" / "data"
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score

## 1. How much of the label is in the justification?

In [ ]:
train = pd.read_parquet(DATA / "train.parquet",
                        columns=["bti_reference", "start_date", "language", "description",
                                 "classification_justification", "heading"])
train = train.sort_values(["start_date", "bti_reference"]).reset_index(drop=True)
just = train["classification_justification"].fillna("")
train["just_names_heading"] = [h in j for h, j in zip(train["heading"], just)]
train["first_number"] = just.str.extract(r"\b(\d{4})\b", expand=False)
print("justification contains the heading:", round(train["just_names_heading"].mean(), 3))
print("first four-digit number = heading:   ", round((train["first_number"] == train["heading"]).mean(), 3))
train.groupby("language")["just_names_heading"].agg(["mean", "size"]).sort_values("size", ascending=False).head(6).round(3)

Read three justifications in a language you know, next to the heading. Where does the heading appear?

In [ ]:
for _, row in train[train["language"] == "en"].sample(3, random_state=1).iterrows():
    print(row["heading"], "|", row["classification_justification"][:300].replace("\n", " "), "\n")

## 2. A model with and without the justification

We train the same linear text classifier on the 50,000-decision sample, once on the description alone and once on description + justification. Then we evaluate the second model twice on 2022–2023: with the justification (as in a careless validation) and without it (as on the test set, where the column does not exist).

In [ ]:
sample = pd.read_parquet(DATA / "train_sample.parquet")
fit = (sample["start_date"] < "2022-01-01").to_numpy()
val = ~fit
y = sample["heading"].to_numpy()
desc = sample["description"].fillna("")
desc_just = desc + " " + sample["classification_justification"].fillna("")


def fit_and_score(fit_text, val_text):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    model = SGDClassifier(loss="hinge", alpha=1e-5, random_state=0, n_jobs=-1)
    model.fit(vec.fit_transform(fit_text[fit]), y[fit])
    return accuracy_score(y[val], model.predict(vec.transform(val_text[val])))


results = {
    "description only": fit_and_score(desc, desc),
    "with justification, validated with it": fit_and_score(desc_just, desc_just),
    "with justification, validated without it (as on the test set)": fit_and_score(desc_just, desc),
}
pd.Series(results, name="accuracy 2022-2023").round(3)

**Question.** Which number would a careless notebook report, and which number will the leaderboard show?

## 3. Lookups by description: leaky, past-only and as-of

In [ ]:
g = train.groupby("description")
train["desc_prev_n"] = g.cumcount()
train["desc_prev_heading"] = g["heading"].shift(1)
train["days_since_prev"] = (train["start_date"] - g["start_date"].shift(1)).dt.days
repeat = train["desc_prev_n"] > 0
print("share of decisions repeating an earlier description:", round(repeat.mean(), 3))
print("same day as the previous one:", round((train.loc[repeat, "days_since_prev"] == 0).mean(), 3))
print("previous heading = heading:   ", round((train["desc_prev_heading"] == train["heading"])[repeat].mean(), 3))

In [ ]:
cut = pd.Timestamp("2022-01-01")
past, later = train[train["start_date"] < cut], train[train["start_date"] >= cut]
most_frequent = train.groupby("description")["heading"].agg(lambda h: h.value_counts().index[0])
as_of = past.groupby("description")["heading"].last()
like_test = ~later["description"].isin(past["description"])
rows = []
for name, pred in {"leaky (all rows)": later["description"].map(most_frequent),
                   "past only (expanding)": later["desc_prev_heading"],
                   "as of 2022-01-01": later["description"].map(as_of)}.items():
    for rows_name, mask in [("all 2022-2023", slice(None)), ("test-like", like_test)]:
        p, yy = pred[mask], later.loc[mask, "heading"]
        rows.append({"lookup": name, "rows": rows_name, "answered": p.notna().mean(), "correct": (p == yy).mean()})
pd.DataFrame(rows).round(3)

**Questions.**

1. Why is the leaky lookup always right?
2. The test set of the leaderboard contains no description that also occurs in `train.parquet`. Which row of the table describes what a lookup would contribute on the leaderboard?
3. Would you still use `desc_prev_heading` in a deployed service (Session 16), where a new request can repeat an old description? Why is it legitimate there?

## 4. Decision

Write down, for each candidate feature, whether it may enter the leaderboard model: `classification_justification`, `keywords`, `desc_prev_n`, `desc_prev_heading`, `start_date` (year, month), `end_date`, the text statistics. For each, state the moment at which its value is known.

## Exercises

1. Repeat Section 2 with `keywords` instead of the justification. How large is the effect?
2. Build a random split instead of the time split in Section 2 (description only). Is the accuracy higher? Explain with the repeated descriptions.